# Copper Module 1 — V26: EDA và kiểm định thống kê chuyên sâu

**Mục tiêu:** kiểm toán dữ liệu gốc và các giả thuyết về chuỗi thời gian trước khi diễn giải kết quả mô hình, **không train lại dự báo**.

**Run All Kaggle:** đưa `dataset_9_commodities_daily.csv` vào Kaggle Input. Tất cả bảng CSV, PNG, manifest và ZIP được ghi vào `/kaggle/working/copper_v26_eda/`.

**Không leakage:** Mọi thống kê để ra quyết định chọn features/độ trễ chỉ dùng **2.195 dòng đầu** (`int(2710*0.9*0.9)`), khớp EDA V20. Phân tích toàn chuỗi, nếu có, chỉ là mô tả hậu nghiệm, tuyệt đối không sử dụng để chọn mô hình hoặc đặc trưng. p-values từ chuỗi thời gian cần được đọc thận trọng khi có phụ thuộc/heteroskedasticity.

**Phân biệt:** kiểm định tương quan, tính dừng, tự tương quan và Granger *không chứng minh* hiệu quả dự báo ngoài mẫu hoặc quan hệ nhân quả kinh tế.

In [ ]:
import os, glob, json, hashlib, warnings, zipfile, sys, platform
from pathlib import Path
import numpy as np, pandas as pd, scipy
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests, coint, zivot_andrews
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch, breaks_cusumolsresid
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
warnings.filterwarnings('ignore', category=FutureWarning)
OUT=Path('/kaggle/working/copper_v26_eda') if Path('/kaggle/working').exists() else Path('/mnt/data/copper_v26_eda')
OUT.mkdir(parents=True,exist_ok=True)
FEATURES=['Copper','Aluminium','Tin','WTI','Gold','Silver']
LAGS=[1,2,5,10,22,50]
EXPECTED_SHA='9580ede12d474c8f990556842d551dac176b3c16b2d6a4bffb7dc41dd54bf600'
files=glob.glob('/kaggle/input/**/dataset_9_commodities_daily.csv',recursive=True)+glob.glob('/mnt/data/**/dataset_9_commodities_daily.csv',recursive=True)
if not files: raise FileNotFoundError('Thiếu dataset_9_commodities_daily.csv. Hãy Add Data vào Kaggle rồi Run All.')
DATA=Path(files[0]); digest=hashlib.sha256(DATA.read_bytes()).hexdigest()
print('Dataset:',DATA,'SHA256:',digest)
if digest != EXPECTED_SHA: raise AssertionError('Dataset SHA256 khác bộ V20–V24; dừng để tránh so sánh sai.')
df=pd.read_csv(DATA); df['Date']=pd.to_datetime(df['Date'],errors='raise')
df=df.sort_values('Date').reset_index(drop=True)
missing_cols=set(FEATURES)-set(df.columns)
assert not missing_cols, f'Thiếu cột: {missing_cols}'
assert len(df)==2710 and df.Date.is_unique and df.Date.is_monotonic_increasing
cut=int(len(df)*0.9*0.9)
train=df.iloc[:cut].copy()
ret=train[FEATURES].pct_change(fill_method=None).replace([np.inf,-np.inf],np.nan)
diff=train[FEATURES].diff()
print('Full:',len(df),df.Date.min(),df.Date.max(),'train EDA:',cut,train.Date.iloc[-1])


## A. Kiểm toán chất lượng và thời gian
Kiểm tra thứ tự, trùng ngày, missing, phi hữu hạn, giá không dương, khoảng cách ngày và bất thường lợi suất. **Phát hiện ngoại lệ thống kê không có nghĩa dữ liệu lỗi**; không tự xóa outliers.

In [ ]:
q=[]
for c in FEATURES:
 s=df[c].astype(float)
 q.append(dict(feature=c, n=len(s), missing=int(s.isna().sum()), nonfinite=int((~np.isfinite(s)).sum()), nonpositive=int((s<=0).sum()), unique=int(s.nunique()), min=s.min(), max=s.max(), median=s.median(), mean=s.mean(), std=s.std()))
quality=pd.DataFrame(q);quality.to_csv(OUT/'01_quality.csv',index=False)
gaps=df.Date.diff().dt.days.dropna()
gapdist=gaps.value_counts().sort_index().rename_axis('gap_days').reset_index(name='count');gapdist.to_csv(OUT/'01_date_gaps.csv',index=False)
quality_checks={'duplicate_dates':int(df.Date.duplicated().sum()),'date_start':str(df.Date.min().date()),'date_end':str(df.Date.max().date()),'n_rows':len(df),'eda_cut':cut,'max_calendar_gap_days':float(gaps.max())}
pd.Series(quality_checks).to_csv(OUT/'01_quality_summary.csv')
display(quality);display(gapdist.tail(12))


## B. Phân phối lợi suất và outliers (train-only)
Sử dụng simple returns `P_t/P_{t-1}-1` tương thích EDA V20, có thêm log returns khi toàn bộ giá dương. Jarque–Bera H₀: phân phối có skew=0, excess kurtosis=0 (normality); p nhỏ bác bỏ tính chuẩn, **không kết luận mô hình phải dùng loss cụ thể**.

In [ ]:
d=[]
for c in FEATURES:
 x=ret[c].dropna(); med=x.median();mad=np.median(np.abs(x-med))
 robust_z=0.67448975*(x-med)/mad if mad>0 else pd.Series(np.nan,index=x.index)
 jb=stats.jarque_bera(x)
 d.append(dict(feature=c,n=len(x),mean=x.mean(),std=x.std(),skew=stats.skew(x),excess_kurtosis=stats.kurtosis(x),q01=x.quantile(.01),median=x.median(),q99=x.quantile(.99),extreme_robust_z_gt_3_5=int((np.abs(robust_z)>3.5).sum()),jarque_bera_stat=float(jb.statistic),jarque_bera_p=float(jb.pvalue)))
returns_stats=pd.DataFrame(d);returns_stats.to_csv(OUT/'02_returns_distribution_JB.csv',index=False);display(returns_stats)
fig,ax=plt.subplots(figsize=(12,4));ax.plot(train.Date,ret.Copper,lw=.65);ax.set_title('Copper returns — train EDA only');ax.set_xlabel('Date');ax.set_ylabel('Simple return');fig.tight_layout();fig.savefig(OUT/'02_copper_returns.png',dpi=150);plt.close(fig)
fig,ax=plt.subplots(figsize=(10,4));ax.hist(ret.Copper.dropna(),bins=65,density=True);ax.set_title('Copper return histogram — train EDA only');fig.tight_layout();fig.savefig(OUT/'02_copper_returns_hist.png',dpi=150);plt.close(fig)


## C. Kiểm định tính dừng — ADF + KPSS (train-only)
- **ADF H₀:** tồn tại nghiệm đơn vị; p nhỏ là bằng chứng chống H₀.
- **KPSS H₀:** chuỗi dừng quanh mức/trend tùy regression; p nhỏ là bằng chứng chống H₀.

Chạy trên **mức giá** và **sai phân**; returns được kiểm tra thêm. Kết quả ADF/KPSS không tự chứng minh quá trình là random walk hay quan hệ nhân quả. Lựa chọn `regression='c'` để khảo sát level/returns và công bố rõ bên dưới.

In [ ]:
station=[]
for c in FEATURES:
 for typ,s in [('level',train[c]),('difference',diff[c]),('simple_return',ret[c])]:
  x=pd.to_numeric(s,errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
  r={'feature':c,'transform':typ,'n':len(x)}
  try:
   aa=adfuller(x,autolag='AIC',regression='c');r.update(adf_stat=aa[0],adf_p=aa[1],adf_lags=aa[2])
  except Exception as e:r['adf_error']=str(e)
  try:
   kk=kpss(x,regression='c',nlags='auto');r.update(kpss_stat=kk[0],kpss_p=kk[1],kpss_lags=kk[2])
  except Exception as e:r['kpss_error']=str(e)
  station.append(r)
station=pd.DataFrame(station);station.to_csv(OUT/'03_adf_kpss.csv',index=False);display(station)


## D. Kiểm định phụ thuộc thời gian và volatility clustering
- Ljung–Box H₀: không có tự tương quan đến lag chỉ định (kiểm trên returns).
- Ljung–Box của returns²: không có tự tương quan của bình phương returns đến lag chỉ định.
- ARCH-LM H₀: không có hiệu ứng ARCH tại lag quy định. Đây **không phải bằng chứng trực tiếp mô hình GARCH sẽ tốt hơn**.

In [ ]:
diagnostics=[]
for c in FEATURES:
 x=ret[c].dropna()
 for lag in [5,10,22]:
  if len(x)<lag+10:continue
  lb=acorr_ljungbox(x,lags=[lag],return_df=True).iloc[0]
  lb2=acorr_ljungbox(x**2,lags=[lag],return_df=True).iloc[0]
  ar=het_arch(x,nlags=lag)
  diagnostics.append(dict(feature=c,lag=lag,lb_stat=lb.lb_stat,lb_p=lb.lb_pvalue,lb_sq_stat=lb2.lb_stat,lb_sq_p=lb2.lb_pvalue,arch_lm_stat=ar[0],arch_lm_p=ar[1]))
diagnostics=pd.DataFrame(diagnostics);diagnostics.to_csv(OUT/'04_lb_arch.csv',index=False);display(diagnostics)
fig,ax=plt.subplots(figsize=(12,4));ax.plot(train.Date,ret.Copper.rolling(22).std(),lw=.9);ax.set_title('Copper rolling std (22 sessions) — train EDA');fig.tight_layout();fig.savefig(OUT/'04_copper_rolling_volatility.png',dpi=150);plt.close(fig)


## E. Correlation level vs. lagged return — điều chỉnh đa kiểm định
Tương quan Spearman và Pearson của **mức giá** là đồng thời, có nguy cơ spurious correlation do xu hướng chung. Đối với mỗi ngoại sinh, ghép `r_feature(t−lag)` với `r_Copper(t)` và chạy 30 phép kiểm định lagged Spearman. **H₀:** không có tương quan thứ hạng trong cặp đang kiểm; p từ Spearman là xấp xỉ và không xử lý đầy đủ tự tương quan. Áp dụng BH-FDR trên **30 phép thử định trước**; không sử dụng p-value để khẳng định predictability. Kết luận về feature phải dựa validation/ablation.

In [ ]:
level_corr=train[FEATURES].corr('spearman');level_corr.to_csv(OUT/'05_spearman_levels_train.csv')
rows=[]
for feature in FEATURES[1:]:
 for lag in LAGS:
  pair=pd.DataFrame({'target':ret.Copper,'driver':ret[feature].shift(lag)}).dropna()
  pr=stats.pearsonr(pair.driver,pair.target)
  sr=stats.spearmanr(pair.driver,pair.target)
  rows.append(dict(feature=feature,lag=lag,n=len(pair),pearson_r=float(pr.statistic),pearson_p=float(pr.pvalue),spearman_r=float(sr.statistic),spearman_p=float(sr.pvalue)))
lagged=pd.DataFrame(rows)
lagged['spearman_p_fdr_bh']=multipletests(lagged.spearman_p,method='fdr_bh')[1]
lagged['pearson_p_fdr_bh']=multipletests(lagged.pearson_p,method='fdr_bh')[1]
lagged.to_csv(OUT/'05_lagged_returns_correlations_FDR.csv',index=False)
display(level_corr.round(3));display(lagged.round(5))
for key in ['pearson_r','spearman_r']:
 mat=lagged.pivot(index='feature',columns='lag',values=key)
 fig,ax=plt.subplots(figsize=(9,4)); im=ax.imshow(mat,aspect='auto',vmin=-.15,vmax=.15,cmap='coolwarm')
 ax.set_xticks(range(len(mat.columns)),mat.columns);ax.set_yticks(range(len(mat.index)),mat.index)
 for i in range(mat.shape[0]):
  for j in range(mat.shape[1]):ax.text(j,i,f'{mat.iloc[i,j]:+.3f}',ha='center',va='center',fontsize=8)
 ax.set_title(f'Lagged returns {key} (train only)');fig.colorbar(im);fig.tight_layout();fig.savefig(OUT/f'05_{key}_heatmap.png',dpi=150);plt.close(fig)


## F. Granger predictive-content test trên returns (train-only)
**H₀:** các lag quá khứ của feature **không thêm khả năng giải thích tuyến tính** cho Copper return ngoài lag Copper. Chạy lag 1/2/5/10 với regression bivariate và kiểm F-test. Chỉ thực hiện nếu returns có bằng chứng tương đối phù hợp với tính dừng; nếu không, đặt trạng thái `not_run_stationarity`. BH-FDR cho tất cả kết quả đã chạy. **Granger không chứng minh nhân quả, không kiểm chứng deep learning và không thay thế out-of-sample validation.**

In [ ]:
gc=[]
station_returns=station[station['transform']=='simple_return'].set_index('feature')
for f in FEATURES[1:]:
 safe=all((pd.notna(station_returns.loc[c,'adf_p']) and station_returns.loc[c,'adf_p']<0.05 and pd.notna(station_returns.loc[c,'kpss_p']) and station_returns.loc[c,'kpss_p']>=0.05) for c in ['Copper',f])
 for lag in [1,2,5,10]:
  record=dict(feature=f,lag=lag,status='not_run_stationarity' if not safe else 'ok')
  if safe:
   pair=pd.DataFrame({'target':ret.Copper,'exog':ret[f]}).dropna()
   try:
    out=grangercausalitytests(pair[['target','exog']],maxlag=lag,verbose=False)
    test=out[lag][0]['ssr_ftest'];record.update(f_stat=float(test[0]),p_value=float(test[1]),n=len(pair))
   except Exception as exc:record.update(status='error',error=str(exc))
  gc.append(record)
gc=pd.DataFrame(gc)
mask=gc.status.eq('ok') & gc.p_value.notna() if 'p_value' in gc else np.zeros(len(gc),dtype=bool)
if mask.sum():gc.loc[mask,'p_fdr_bh']=multipletests(gc.loc[mask,'p_value'],method='fdr_bh')[1]
gc.to_csv(OUT/'06_granger_returns.csv',index=False);display(gc)


## G. Đa cộng tuyến & kiểm tra biến cấu trúc
- VIF đánh giá phụ thuộc tuyến tính giữa các ngoại sinh, chỉ là chỉ số mô tả (không là bằng chứng bỏ features cho NN).
- CUSUM đối với mean returns dùng hồi quy hằng số để kiểm độ ổn định trung bình có điều kiện: **H₀:** hệ số hồi quy ổn định; có thể không nhạy với biến động volatility/tail. Không coi đây là chứng minh concept drift.
- Zivot–Andrews kiểm tra **H₀: unit root không có break**, đối lập với chuỗi trend-stationary có một break nội sinh. Chỉ số break là vị trí ứng viên do phép kiểm định tìm; **không phải ngày xác nhận biến cố kinh tế**.
- Brown–Forsythe (Levene, center=median) trên hai nửa train *định trước* kiểm tra H₀: phương sai returns hai giai đoạn bằng nhau. Kết quả không xác định vị trí break tối ưu.
- Cointegration Engle–Granger mức giá chỉ diễn giải khi cả hai level có dấu hiệu I(1) phù hợp; kết quả là thăm dò và kiểm định nhiều cặp cần FDR.

In [ ]:
vif_rows=[]
for transform,X in [('levels',train[FEATURES[1:]].copy()),('returns',ret[FEATURES[1:]].dropna().copy())]:
 X=X.astype(float).dropna()
 for i,col in enumerate(X.columns):
  try:v=float(variance_inflation_factor(sm.add_constant(X).values,i+1))
  except Exception:v=np.nan
  vif_rows.append({'transform':transform,'feature':col,'vif':v})
vif_df=pd.DataFrame(vif_rows);vif_df.to_csv(OUT/'07_vif.csv',index=False);display(vif_df)
cus=[]
for f in FEATURES:
 x=ret[f].dropna();fit=sm.OLS(x.to_numpy(),np.ones((len(x),1))).fit()
 t=breaks_cusumolsresid(fit.resid,ddof=1)
 cus.append({'feature':f,'cusum_stat':float(t[0]),'cusum_p':float(t[1])})
cus=pd.DataFrame(cus);cus['p_fdr_bh']=multipletests(cus.cusum_p,method='fdr_bh')[1]
cus.to_csv(OUT/'07_cusum_mean_returns.csv',index=False);display(cus)
# Structural-break diagnostics: Zivot–Andrews + predeclared variance comparison
za_rows=[]; variance_rows=[]
for f in FEATURES:
 x=train[f].astype(float).to_numpy()
 try:
  za=zivot_andrews(x,maxlag=12,regression='ct',autolag='AIC',trim=0.15)
  b=int(za[4]); za_rows.append({'feature':f,'za_stat':float(za[0]),'za_p':float(za[1]),'lag_used':int(za[3]),'break_index_train':b,'break_date_train':str(train.Date.iloc[b].date()),'regression':'ct','status':'ok'})
 except Exception as e:za_rows.append({'feature':f,'status':'error','error':str(e)})
 rr=ret[f].dropna().to_numpy(); nrr=len(rr); m=nrr//2
 bf=stats.levene(rr[:m],rr[m:],center='median')
 variance_rows.append({'feature':f,'n_first':m,'n_second':nrr-m,'first_period_end':str(train.Date.iloc[m].date()),'second_period_start':str(train.Date.iloc[m+1].date()),'std_first':float(np.std(rr[:m],ddof=1)),'std_second':float(np.std(rr[m:],ddof=1)),'brown_forsythe_stat':float(bf.statistic),'brown_forsythe_p':float(bf.pvalue)})
za_df=pd.DataFrame(za_rows)
if 'za_p' in za_df:
 m=za_df.za_p.notna()
 if m.any():za_df.loc[m,'p_fdr_bh']=multipletests(za_df.loc[m,'za_p'],method='fdr_bh')[1]
za_df.to_csv(OUT/'07_zivot_andrews_break_unit_root.csv',index=False)
variance_df=pd.DataFrame(variance_rows);variance_df['p_fdr_bh']=multipletests(variance_df.brown_forsythe_p,method='fdr_bh')[1]
variance_df.to_csv(OUT/'07_brown_forsythe_variance_halves.csv',index=False)
display(za_df);display(variance_df)
coin=[]
for f in FEATURES[1:]:
 def looks_I1(name):
  a=station.set_index(['feature','transform'])
  return (pd.notna(a.loc[(name,'level'),'adf_p']) and pd.notna(a.loc[(name,'difference'),'adf_p']) and a.loc[(name,'level'),'adf_p']>0.05 and a.loc[(name,'difference'),'adf_p']<0.05)
 if looks_I1('Copper') and looks_I1(f):
  try: t,pv,_=coint(train.Copper,train[f],trend='c',autolag='aic');coin.append({'feature':f,'status':'tested_exploratory','coint_t':t,'p_value':pv})
  except Exception as e:coin.append({'feature':f,'status':'error','error':str(e)})
 else:coin.append({'feature':f,'status':'not_tested_I1_not_supported'})
coin=pd.DataFrame(coin)
if 'p_value' in coin:
 m=coin.p_value.notna()
 if m.any():coin.loc[m,'p_fdr_bh']=multipletests(coin.loc[m,'p_value'],method='fdr_bh')[1]
coin.to_csv(OUT/'07_cointegration_exploratory.csv',index=False);display(coin)


## H. Tổng kết tự động, khả năng tái lập
**Không tự động chọn feature theo test.** Bảng JSON ghi rõ số hàng, tập train EDA, versions, SHA256 và đường dẫn. Các cảnh báo khoa học bắt buộc: thiếu as-of availability từng thị trường; multiple testing không thay thế validation; dữ liệu giá level có thể không dừng; Granger không phải nhân quả; nghiên cứu thăm dò không bảo đảm tổng quát hóa.

In [ ]:
summary={
 'data_file':str(DATA),'sha256':digest,'all_rows':len(df),'eda_train_rows':cut,'eda_train_last_date':str(train.Date.iloc[-1].date()),
 'date_start':str(df.Date.iloc[0].date()),'date_end':str(df.Date.iloc[-1].date()),
 'features':FEATURES,'lags':LAGS,'alpha':0.05,'missing_total':int(quality.missing.sum()),'duplicate_dates':quality_checks['duplicate_dates'],
 'structural_break_methods':['CUSUM mean returns','Zivot-Andrews level break-adjusted unit root','Brown-Forsythe fixed two-half variance comparison'],
 'warnings':['Train-only EDA is NOT independent future-test evidence','No as-of-availability audit from daily aligned price dates','Correlations and Granger tests do NOT prove causal relations or OOS improvement','Sample autocorrelation may affect standard correlation p-values'],
 'versions':{'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__,'statsmodels':sm.__version__},
 'outputs': sorted(x.name for x in OUT.iterdir())}
(OUT/'manifest.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
zip_path=OUT.parent/'copper_v26_comprehensive_eda_results.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for f in OUT.iterdir():
  if f.is_file():z.write(f,arcname=f.name)
print('DONE:',zip_path,'files:',len(summary['outputs'])+1)
